# Dataset-MoE NIDS — complete Colab training and evaluation

Run this notebook top to bottom for the finalized four-dataset `moe_dataset_soft` comparison against `moe_nids_diagnostic` notebook 04's pooled MLP. It uses the same four NF-v3 datasets, 47-feature order, label vocabulary, split algorithm/seed, pooled train-fitted scaling, batch size, optimizer family, class weighting, and validation-based stopping. The architectural difference is four softly mixed dataset experts instead of one pooled classification head.

Before running: select a GPU runtime, add a Colab Secret named `GITHUB_TOKEN` with read access to the private repository, and place the selected datasets below `MyDrive/NIDS_datasets/`.


In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = "selimsidan"
GITHUB_REPO = "dataset_moe_nids"
GITHUB_BRANCH = "main"
GITHUB_SECRET_NAME = "GITHUB_TOKEN"

DRIVE_DATA_DIR = "/content/drive/MyDrive/NIDS_datasets"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs"

# Production uses every row through signed disk-backed splits. Use
# in_memory_smoke once first to validate the pipeline on small samples.
EXECUTION_MODE = "out_of_core_full"  # out_of_core_full | in_memory_smoke

# Any 2, 3, or 4 of these schema-compatible NF-v3 datasets.
ACTIVE_DATASETS = [
    "NF-UNSW-NB15-v3",
    "NF-ToN-IoT-v3",
    "NF-BoT-IoT-v3",
    "NF-CICIDS2018-v3",
]

# Finalized four-expert comparison against diagnostic notebook 04's
# pooled MLP. Keep these settings unchanged for the primary run.
ARCHITECTURE = "moe_dataset_soft"
RUN_NAME = "nfv3_4way_moe_soft_comparable_seed0_v1"
SEED = 0

# The shared encoder matches the pooled MLP's dense widths: 47→128→64.
# Four separate linear 64→classes experts add only the specialization
# required by MoE, rather than giving it a much larger deep expert bank.
LATENT_DIM = 64
ENCODER_HIDDEN_DIMS = [128]
EXPERT_HIDDEN_DIMS = []
DROPOUT = 0.2
GATE_SUPERVISION = "none"  # no training-time dataset-ID advantage
LAMBDA_BALANCE = 0.1

EPOCHS_A = 30  # pooled representation pretraining
EPOCHS_B = 10  # dataset-head warm-start
EPOCHS_C = 30  # joint soft-gated training; validation early stopping
BATCH_SIZE = 512
FORCE_RESTART = False  # False resumes only contract-compatible checkpoints
RUN_TESTS = True
# ==================================================================


## 1. Secure checkout and environment setup

The GitHub token is supplied through a temporary HTTP header. It is never embedded in the repository URL or persisted in Git configuration.


In [ ]:
import base64, os, subprocess, sys
from pathlib import Path
try:
    from google.colab import drive, userdata
except ImportError as exc:
    raise RuntimeError("This notebook is intended for Google Colab.") from exc
drive.mount("/content/drive")
token = userdata.get(GITHUB_SECRET_NAME)
if not token:
    raise RuntimeError(f"Add {GITHUB_SECRET_NAME} in Colab Secrets and grant notebook access.")
auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
git_env = os.environ | {
    "GIT_CONFIG_COUNT": "1",
    "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
    "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
}
repo_url = f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git"
repo_dir = Path("/content") / GITHUB_REPO
if (repo_dir / ".git").is_dir():
    subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", GITHUB_BRANCH], env=git_env, check=True)
else:
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, "--single-branch", repo_url, str(repo_dir)], env=git_env, check=True)
git_env.clear()
token = auth = None
os.chdir(repo_dir)
os.environ["NIDS_DRIVE_BASE"] = DRIVE_DATA_DIR
os.environ["NIDS_OUTPUT_DIR"] = DRIVE_OUTPUT_DIR
os.environ["NIDS_SCRATCH_DIR"] = "/content/dataset_moe_nids_scratch"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
print("Repository:", repo_dir)
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. Preflight and tests

Full-data mode requires two to four registered single-file NF-v3 datasets with the identical confirmed 47-feature schema. Split membership and pooled preprocessing are reused only when their signed contracts match.


In [ ]:
import torch
from data.registry import get_spec
from training.config import load_config
if EXECUTION_MODE not in {"out_of_core_full", "in_memory_smoke"}:
    raise ValueError("Unknown EXECUTION_MODE")
if ARCHITECTURE != "moe_dataset_soft":
    raise ValueError("This finalized notebook run is specifically for moe_dataset_soft")
if len(ACTIVE_DATASETS) != len(set(ACTIVE_DATASETS)):
    raise ValueError("ACTIVE_DATASETS contains duplicates")
if EXECUTION_MODE == "out_of_core_full" and not 2 <= len(ACTIVE_DATASETS) <= 4:
    raise ValueError("Full-data mode requires a 2-way, 3-way, or 4-way combination")
if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Select Runtime → Change runtime type → GPU and reconnect.")
missing = {}
for name in ACTIVE_DATASETS:
    spec = get_spec(name)
    existing = [Path(path) for path in spec.paths if Path(path).is_file()]
    if not existing:
        missing[name] = spec.paths
if missing:
    raise FileNotFoundError("Missing selected datasets:\n" + "\n".join(f"  {k}: {v}" for k, v in missing.items()))
if EXECUTION_MODE == "out_of_core_full":
    aliases = [get_spec(name).feature_alias for name in ACTIVE_DATASETS]
    if any(get_spec(name).kind != "file" for name in ACTIVE_DATASETS) or any(value != aliases[0] for value in aliases[1:]):
        raise ValueError("Full-data combinations must use schema-compatible single-file NF-v3 datasets")
    assert len(aliases[0]) == 47
preview = load_config("config/default.yaml", ["data.active_datasets=[" + ",".join(ACTIVE_DATASETS) + "]"])
selected_classes = {"Benign"}
for name in ACTIVE_DATASETS:
    selected_classes.update(value for value in preview["data"]["label_mapping"][name].values() if value is not None)
num_classes = len(selected_classes)
encoder_params = (47 + 1) * 128 + (128 + 1) * 64
expert_params = len(ACTIVE_DATASETS) * (64 + 1) * num_classes
gate_params = (64 + 1) * len(ACTIVE_DATASETS)
moe_params = encoder_params + expert_params + gate_params
diagnostic_mlp_params = (47 + 1) * 128 + 2 * 128 + (128 + 1) * 64 + 2 * 64 + (64 + 1) * num_classes
print("GPU:", torch.cuda.get_device_name(0))
print("Combination:", ACTIVE_DATASETS)
print("Architecture:", ARCHITECTURE)
print("Encoder/expert:", [47, *ENCODER_HIDDEN_DIMS, LATENT_DIM], "+ four", [LATENT_DIM, *EXPERT_HIDDEN_DIMS, "classes"], "experts")
print(f"Selected classes: {num_classes}; deployed parameters: MoE={moe_params:,}, diagnostic pooled MLP≈{diagnostic_mlp_params:,}, ratio={moe_params/diagnostic_mlp_params:.2f}x")
print("Execution mode:", EXECUTION_MODE, "| run:", RUN_NAME)


In [ ]:
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)
else:
    print("Tests skipped by configuration.")


## 3. Train Stage A → B → C and evaluate

Stage A is retained because shared columns do not provide a learned shared representation: it trains the same-sized `47→128→64` pooled encoder before expert specialization, preventing random gate/expert cold-start. Stage B warm-starts four linear dataset heads, and Stage C jointly trains the task-driven soft gate with no dataset-ID auxiliary loss. The run contract prevents incompatible checkpoint reuse, and progress is checkpointed atomically after every epoch.


In [ ]:
overrides = [
    f"run_name={RUN_NAME}",
    f"seed={SEED}",
    f"architecture={ARCHITECTURE}",
    "data.active_datasets=[" + ",".join(ACTIVE_DATASETS) + "]",
    "training.device=cuda",
    f"training.force_restart={str(FORCE_RESTART).lower()}",
    f"training.epochs_a={EPOCHS_A}",
    f"training.epochs_b={EPOCHS_B}",
    f"training.epochs_c={EPOCHS_C}",
    f"training.batch_size={BATCH_SIZE}",
    f"model.latent_dim={LATENT_DIM}",
    "model.encoder.hidden_dims=[" + ",".join(map(str, ENCODER_HIDDEN_DIMS)) + "]",
    "model.expert.hidden_dims=[" + ",".join(map(str, EXPERT_HIDDEN_DIMS)) + "]",
    f"model.encoder.dropout={DROPOUT}",
    f"model.expert.dropout={DROPOUT}",
    f"training.stage_c.gate_supervision={GATE_SUPERVISION}",
    "training.stage_c_unfreeze=all",
    f"load_balance.lambda_balance={LAMBDA_BALANCE}",
]
module = "training.ooc_run" if EXECUTION_MODE == "out_of_core_full" else "training.run"
cmd = [sys.executable, "-m", module, "--config", "config/default.yaml"]
if EXECUTION_MODE == "in_memory_smoke":
    cmd += ["--mode", "smoke"]
for override in overrides:
    cmd += ["--set", override]
print("Launching:", module)
subprocess.run(cmd, check=True, env=os.environ.copy())
print("Training and evaluation completed.")


## 4. Review persisted detailed results

These tables are loaded back from Drive. `ALL` is the combined test population; the other rows are dataset-of-origin results. Per-class metrics include exact TP/FP/FN/TN, specificity, NPV, false-positive/negative rates, and class-balanced accuracy.


In [ ]:
import pandas as pd
from IPython.display import display
result_dir = Path(DRIVE_OUTPUT_DIR) / "results" / RUN_NAME
overall = pd.read_csv(result_dir / "Overall_Metrics.csv")
per_class = pd.read_csv(result_dir / "Per_Class_Metrics.csv")
print("=== Overall and per-origin metrics ===")
display(overall)
print("=== Native per-class metrics, weakest F1 first ===")
if "is_native_class" in per_class.columns:
    display(per_class[per_class["is_native_class"]].sort_values(["origin", "f1", "support"]).reset_index(drop=True))
else:
    display(per_class.sort_values(["f1", "support"]).reset_index(drop=True))
for filename in ["Gate_By_Dataset.csv", "Expert_Utilization.csv", "Confusion_Matrix.csv"]:
    path = result_dir / filename
    if path.is_file():
        print(f"=== {filename} ===")
        display(pd.read_csv(path))
print("Detailed artifacts:", result_dir)
print("Checkpoints:", Path(DRIVE_OUTPUT_DIR) / "checkpoints" / RUN_NAME)


## Operating guidance

- First run the selected combination with `EXECUTION_MODE = "in_memory_smoke"` and a smoke-specific `RUN_NAME`.
- Then switch to `out_of_core_full` and a new production `RUN_NAME`.
- Keep the finalized architecture settings unchanged for the primary pooled-MLP comparison; change only `SEED` and `RUN_NAME` for additional seeds.
- Keep `FORCE_RESTART = False` after interruptions. Set it to `True` only when intentionally deleting this run’s stage checkpoints.
- Prepared split artifacts are shared safely across MoE runs because their signatures include source identity, label mapping, feature schema, split policy, and seed.
